# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PundarikakshNTripathi/FlyRankAI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

We will establish a transparent baseline heuristic:

**Rule:** A page is highly prioritized if it is 'Stale and Visible' or 'Decaying on Page 1'.

- `stale_visible`: age > 180 days AND impressions_90d > 500
- `page_1_decay`: position <= 10 AND age > 180 days

We score 1 for each condition met.

## 2. Build the ranked queue

Scoring and ranking.

In [1]:
import pandas as pd
import os

# Ensure path works whether run from root or work/notebooks/
data_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(data_path)

# Compute heuristic baseline scores
stale_visible = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)).astype(int)
page_1_decay = ((df['avg_position'] > 0) & (df['avg_position'] <= 10) & (df['content_age_days'] >= 180)).astype(int)

df['baseline_score'] = stale_visible + page_1_decay
df['baseline_reason'] = ''
df.loc[stale_visible == 1, 'baseline_reason'] += 'stale_visible; '
df.loc[page_1_decay == 1, 'baseline_reason'] += 'page_1_decay; '

queue = df[df['baseline_score'] > 0].sort_values(by='baseline_score', ascending=False)

out_dir = '../../outputs'
if not os.path.exists('../../outputs') and os.path.exists('outputs'):
    out_dir = 'outputs'
os.makedirs(out_dir, exist_ok=True)

queue[['client_id', 'content_id', 'baseline_score', 'baseline_reason']].head(100).to_csv(f'{out_dir}/baseline_queue.csv', index=False)
print(f"Baseline queue saved. Found {len(queue)} pages matching baseline rules.")

Baseline queue saved. Found 7090 pages matching baseline rules.


## 3. Top-20 review

The top 20 pages consistently show high impressions and old update dates. However, some might be evergreen pages that don't actually need a refresh, highlighting the limitation of a simple heuristic.

## 4. Weak picks + leakage check

Weak picks: Evergreen glossary pages that are old but still perfectly accurate. The heuristic cannot differentiate intent.
No leakage found: The rule uses only observable age and historical visibility.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.